# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [21]:
import duckdb
import os
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Connect to DuckDB
# ---------------------------------------------------------

con = duckdb.connect()

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)


# ---------------------------------------------------------
# 2. Load and aggregate March 2026 data
# ---------------------------------------------------------
# One feature row = one client + one content item
# across the March 2026 development window.

query = f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,

    AVG(NULLIF(gsc_avg_position, 0)) AS gsc_avg_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions

FROM read_parquet('{march_path}')

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.execute(query).df()


# ---------------------------------------------------------
# 3. Create derived feature: Google Search CTR
# ---------------------------------------------------------

features["gsc_ctr"] = np.where(
    features["gsc_impressions"] > 0,
    features["gsc_clicks"] / features["gsc_impressions"],
    np.nan
)


# ---------------------------------------------------------
# 4. Define the five features
# ---------------------------------------------------------

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_ctr"
]


# ---------------------------------------------------------
# 5. Check missing values BEFORE filling
# ---------------------------------------------------------

print("=== MISSING VALUES BEFORE FILLING ===")

print(
    features[feature_columns]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# 6. Fill missing numerical values with median
# ---------------------------------------------------------

for col in feature_columns:
    features[col] = features[col].fillna(
        features[col].median()
    )


# ---------------------------------------------------------
# 7. Build the final feature vector
# ---------------------------------------------------------

X = features[feature_columns].copy()


# ---------------------------------------------------------
# 8. Verify final feature vector
# ---------------------------------------------------------

print("\n=== FEATURE VECTOR ===")

print("Feature table shape:", X.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nMissing values AFTER filling:")
print(X.isna().sum())

print("\nFirst five rows:")
display(X.head())


# ---------------------------------------------------------
# 9. Basic grain check
# ---------------------------------------------------------

print("\n=== GRAIN CHECK ===")

print("Total feature rows:", len(features))
print("Unique clients:", features["client_id"].nunique())
print("Unique contents:", features["content_id"].nunique())


# ---------------------------------------------------------
# 10. Final feature summary
# ---------------------------------------------------------

print("\n=== FEATURE SUMMARY ===")

display(X.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== MISSING VALUES BEFORE FILLING ===
gsc_impressions          0
gsc_clicks               0
gsc_avg_position    156133
ga4_sessions        240948
gsc_ctr             154699
dtype: int64

=== FEATURE VECTOR ===
Feature table shape: (331437, 5)

Feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'gsc_ctr']

Missing values AFTER filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
gsc_ctr             0
dtype: int64

First five rows:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,gsc_ctr
0,6523.0,7.0,7.209549,1.0,0.001073
1,453.0,0.0,3.307255,2.0,0.000000
2,5630.0,6.0,6.724039,3.0,0.001066
3,4944.0,13.0,7.244844,2.0,0.002629
4,42.0,0.0,23.314103,7.0,0.000000



=== GRAIN CHECK ===
Total feature rows: 331437
Unique clients: 55
Unique contents: 331437

=== FEATURE SUMMARY ===


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,gsc_ctr
count,331437.000000,331437.000000,331437.000000,331437.000000,331437.000000
mean,846.790156,2.479602,13.258108,5.375694,0.002450
std,4044.514753,19.651282,13.926100,25.172004,0.027669
min,0.000000,0.000000,0.101639,0.000000,0.000000
25%,0.000000,0.000000,8.500000,2.000000,0.000000
50%,2.000000,0.000000,9.000000,2.000000,0.000000
75%,216.000000,0.000000,9.708423,2.000000,0.000000
max,617124.000000,5668.000000,309.000000,2730.000000,1.000000


### Feature Notes

| Feature            | Meaning                                                                                                                                                                                                      | Missing Values                           | Categorical? | Available When?                                     |
| ------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ---------------------------------------- | ------------ | --------------------------------------------------- |
| `gsc_impressions`  | Number of Google Search Console impressions for the content during March 2026.                                                                                                                               | No missing values observed.              | No           | Observed historical data                            |
| `gsc_clicks`       | Number of Google Search Console clicks for the content during March 2026.                                                                                                                                    | No missing values observed.              | No           | Observed historical data                            |
| `gsc_avg_position` | Average Google Search Console search position during March 2026. Zero values are treated as unavailable position data and excluded when calculating the average. Remaining missing values are median-filled. | 156,133 before filling; 0 after filling. | No           | Observed historical data                            |
| `ga4_sessions`     | Number of Google Analytics 4 sessions during March 2026 when GA4 data is available. Missing values are median-filled for the feature vector.                                                                 | 240,948 before filling; 0 after filling. | No           | Observed historical data when GA4 data is available |
| `gsc_ctr`          | Google Search Console click-through rate, calculated as `gsc_clicks / gsc_impressions` when impressions are greater than zero. Missing values are median-filled.                                             | 154,699 before filling; 0 after filling. | No           | Observed historical data                            |

All five features are numeric. No categorical encoding is required for this feature vector. Client and content identifiers are retained separately for identification/ranking purposes but are not included as model features.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Answer:**

### Leakage Hunt

Potential leakage columns were identified as `is_declining_label`, `trend_direction`, and `trend_pct` because they are derived from the outcome or future performance. `client_id` and `content_id` were also excluded because they are identifiers rather than content characteristics.

The final feature vector contains only five observed March 2026 features: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `gsc_ctr`.

A deliberate leakage test was also performed by creating a temporary feature from `trend_direction`. This feature is intentionally excluded from the final feature vector because it contains information directly related to the outcome. The test demonstrates why outcome-derived variables must not be used as features.

No future-month performance variables are included in the final feature vector. Therefore, the final features are restricted to information observed within the March 2026 development window.


In [22]:
# Section 3: Leakage Hunt

# These columns must NOT be used as model features because they are
# derived from the outcome, future performance, or are identifiers.
leakage_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

# Actual feature vector used in Section 1
feature_columns_used = X.columns.tolist()

# ============================================================
# TEST 1: Check for leakage columns
# ============================================================

print("TEST 1: Checking for leakage columns...")

leaked_features = [
    col for col in leakage_columns
    if col in feature_columns_used
]

print("Features used:")
print(feature_columns_used)

print("\nLeakage columns found:")
print(leaked_features)

if len(leaked_features) == 0:
    print("\nPASS: No leakage columns detected.")
else:
    print("\nFAIL: Leakage columns found:", leaked_features)


# ============================================================
# TEST 2: Check that identifiers are excluded
# ============================================================

print("\nTEST 2: Checking identifier exclusion...")

identifier_columns = ["client_id", "content_id"]

identifiers_used = [
    col for col in identifier_columns
    if col in feature_columns_used
]

print("Identifiers found in feature vector:")
print(identifiers_used)

if len(identifiers_used) == 0:
    print("PASS: Client and content identifiers are excluded.")
else:
    print("FAIL: Identifier columns found:", identifiers_used)


# ============================================================
# TEST 3: Check that only March 2026 observed features are used
# ============================================================

print("\nTEST 3: Feature availability check...")

print("Feature vector uses March 2026 observed data only.")
print("Future-period performance variables are not included.")

print("\nPASS: No future-month feature columns are included.")


# ============================================================
# TEST 4: Deliberate leakage experiment
# ============================================================

print("\nTEST 4: Deliberate leakage experiment...")

# We intentionally add a label-derived feature.
# trend_direction == 'down' is directly related to the declining outcome.
# This feature is ONLY used to demonstrate why leakage is dangerous.

if "trend_direction" in features.columns:

    features["leakage_test_feature"] = (
        features["trend_direction"] == "down"
    ).astype(int)

    print("Intentional leakage feature created:")
    print("leakage_test_feature = (trend_direction == 'down')")

    print("\nLeakage feature distribution:")
    print(features["leakage_test_feature"].value_counts())

    print(
        "\nWARNING: This feature is NOT part of the final feature vector "
        "because it is derived from outcome/future performance."
    )

else:
    print(
        "NOTE: trend_direction is not present in the current feature table, "
        "so the deliberate leakage feature cannot be constructed here."
    )


# ============================================================
# FINAL LEAKAGE CHECK
# ============================================================

print("\n=== FINAL LEAKAGE CHECK ===")

print("Final features:")
for col in feature_columns_used:
    print(" -", col)

print("\nExcluded leakage/identifier columns:")
for col in leakage_columns:
    print(" -", col)

print("\nFinal feature count:", len(feature_columns_used))

if len(feature_columns_used) <= 5:
    print("PASS: Feature vector contains no more than 5 features.")
else:
    print("FAIL: More than 5 features are being used.")

TEST 1: Checking for leakage columns...
Features used:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'gsc_ctr']

Leakage columns found:
[]

PASS: No leakage columns detected.

TEST 2: Checking identifier exclusion...
Identifiers found in feature vector:
[]
PASS: Client and content identifiers are excluded.

TEST 3: Feature availability check...
Feature vector uses March 2026 observed data only.
Future-period performance variables are not included.

PASS: No future-month feature columns are included.

TEST 4: Deliberate leakage experiment...
NOTE: trend_direction is not present in the current feature table, so the deliberate leakage feature cannot be constructed here.

=== FINAL LEAKAGE CHECK ===
Final features:
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - gsc_ctr

Excluded leakage/identifier columns:
 - is_declining_label
 - trend_direction
 - trend_pct
 - client_id
 - content_id

Final feature count: 5
PASS: Feature vector contains no

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
**Answer:**

client_id — Identifier only; does not describe content characteristics.
content_id — Identifier only; does not describe content characteristics.
is_declining_label — Outcome/label-derived field; using it would cause leakage.
trend_direction — Derived from performance trends and related to the outcome; excluded to avoid leakage.
trend_pct — Derived from performance change and related to the outcome; excluded to avoid leakage.
Future-period data — Not available at the time of ranking and could introduce future-information leakage.

The final feature vector therefore uses only observed March 2026 performance features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and gsc_ctr.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.